In [0]:
CATALOG_NAME = 'workspace'
SCHEMA_NAME = 'adventureworks'

Helper functions

In [0]:
# check if a table has a description 
def getTableComment(catalogName, schemaName, tableName):
    query = f"SELECT IFF(comment is null, false, true) as tableComment FROM {catalogName}.information_schema.tables WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    tableComment = spark.sql(query)  
    return tableComment.first()["tableComment"]
  
# check if all columns in a table have a description 
def getColumnComment(catalogName, schemaName, tableName):
    query = f"SELECT EVERY(comment is not null) as columnComment FROM {catalogName}.information_schema.columns WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    columnComment = spark.sql(query)   
    return columnComment.first()["columnComment"]

# check if I am the owner of the table
def getAcceptedOwner(catalogName, schemaName, tableName):
    query = f"SELECT IFF(table_owner = 'saskia.vanderwegen@infosupport.com', true, false) as acceptedOwner FROM {catalogName}.information_schema.tables WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    acceptedOwner = spark.sql(query)   
    return acceptedOwner.first()["acceptedOwner"]

# check how many weeks a table is in production
def getWeeksInProduction(catalogName, schemaName, tableName):
    query = f"SELECT FLOOR(DATEDIFF(GETDATE(), created)/7) as weeksInProduction FROM {catalogName}.information_schema.tables WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    weeksInProduction = spark.sql(query)   
    return weeksInProduction.first()["weeksInProduction"]

# Freshness: in elk geval kan je last altered date uit het informatie schema halen. Uitgebreidere checks zijn mogelijk afhankelijk van de freshness implementatie.  
# Has data checks: in elk geval kan je constraints op de tabel checken. Meer uitgebreide DQC's hangt af van de implementatie
# Access specificity: ik weet niet precies wat ze hier willen, maar je kan iig zien wie er toegang tot tabellen heeft, dus er moet wel iets mee te doen zijn. 
# Cost budget tags: Tags zijn eenvoudig te querien via het information schema. Moet je alleen wel iets van cost budgets in tags implementeren natuurlijk.

Determine and persist score details for a given table

In [0]:
def determineTrustScoreDetails(catalogName, schemaName, tableName):
    hasComments = getTableComment(catalogName, schemaName, tableName)
    allColumnsHaveComments = getColumnComment(catalogName, schemaName, tableName)
    hasHumanOwner = getAcceptedOwner(catalogName, schemaName, tableName)
    weeksInProduction = getWeeksInProduction(catalogName, schemaName, tableName)

    upsert_query = f"""
    MERGE INTO workspace.trustmodel.trust_scores AS target
    USING (
        SELECT 
            '{catalogName}' AS catalogName, 
            '{schemaName}' AS schemaName, 
            '{tableName}' AS tableName, 
            '{hasComments}' AS hasComments, 
            '{allColumnsHaveComments}' AS allColumnsHaveComments, 
            '{hasHumanOwner}' AS hasHumanOwner, 
            '{weeksInProduction}' AS weeksInProduction
    ) AS source
    ON target.catalogName = source.catalogName 
       AND target.schemaName = source.schemaName 
       AND target.tableName = source.tableName
    WHEN MATCHED THEN
        UPDATE SET 
            target.hasComments = source.hasComments, 
            target.allColumnsHaveComments = source.allColumnsHaveComments, 
            target.hasHumanOwner = source.hasHumanOwner, 
            target.weeksInProduction = source.weeksInProduction
    WHEN NOT MATCHED THEN
        INSERT (
            catalogName, 
            schemaName, 
            tableName, 
            hasComments, 
            allColumnsHaveComments, 
            hasHumanOwner, 
            weeksInProduction
        )
        VALUES (
            source.catalogName, 
            source.schemaName, 
            source.tableName, 
            source.hasComments, 
            source.allColumnsHaveComments, 
            source.hasHumanOwner, 
            source.weeksInProduction
        )
    """
    spark.sql(upsert_query)

Determine and persist the trust score details for all tables in CATALOG_NAME.SCHEMA_NAME.

In [0]:
tables_query = f"SELECT table_name FROM {CATALOG_NAME}.information_schema.tables WHERE table_schema = '{SCHEMA_NAME}'"
tables_df = spark.sql(tables_query)

for row in tables_df.collect():
    table_name = row["table_name"]
    determineTrustScoreDetails(CATALOG_NAME, SCHEMA_NAME, table_name)

Determine and persist the trust score for all records.

In [0]:
update_query = """
UPDATE workspace.trustmodel.trust_scores
SET trust_score = 15 * CAST(hasComments AS INT) + 15 * CAST(allColumnsHaveComments AS INT) + 16 * CAST(hasHumanOwner AS INT) + 8 * LEAST(weeksInProduction, 5)
"""
spark.sql(update_query)

For debugging purposes, query the results.

In [0]:
%sql
SELECT * FROM workspace.trustmodel.trust_scores